<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Modelo de lenguaje con tokenización por caracteres

### Consigna
- Seleccionar un corpus de texto sobre el cual entrenar el modelo de lenguaje.
- Realizar el pre-procesamiento adecuado para tokenizar el corpus, estructurar el dataset y separar entre datos de entrenamiento y validación.
- Proponer arquitecturas de redes neuronales basadas en unidades recurrentes para implementar un modelo de lenguaje.
- Con el o los modelos que consideren adecuados, generar nuevas secuencias a partir de secuencias de contexto con las estrategias de greedy search y beam search determístico y estocástico. En este último caso observar el efecto de la temperatura en la generación de secuencias.


### Sugerencias
- Durante el entrenamiento, guiarse por el descenso de la perplejidad en los datos de validación para finalizar el entrenamiento. Para ello se provee un callback.
- Explorar utilizar SimpleRNN (celda de Elman), LSTM y GRU.
- rmsprop es el optimizador recomendado para la buena convergencia. No obstante se pueden explorar otros.


In [1]:
import random
import io
import pickle
from cProfile import label

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from tensorflow import keras
from tensorflow.keras import layers
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, LSTM, Embedding, Dropout
from tensorflow.keras.losses import SparseCategoricalCrossentropy

2025-04-23 19:33:57.178699: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-23 19:33:57.189214: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745447637.199072 2045576 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745447637.202078 2045576 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1745447637.211488 2045576 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

### Datos
Utilizaremos como dataset uno disponible en kaggle sobre papers relacionados a AI. Contiene tags y abstracts (parrafo resumen). Voy a utilizar los primeros 10000 titulos + abstracts compilados en un unico texto.


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sumitm004/arxiv-scientific-research-papers-dataset")

print("Path to dataset files:", path)

Path to dataset files: /home/juan/.cache/kagglehub/datasets/sumitm004/arxiv-scientific-research-papers-dataset/versions/2


In [3]:
# With the dataset loaded, we can now read the csv into a pandas dataframe     
df = pd.read_csv(path + '/arXiv_scientific dataset.csv')  

In [4]:
df

,id,title,category,category_code,published_date,updated_date,authors,first_author,summary,summary_word_count
0,cs-9308101v1,Dynamic Backtracking,Artificial Intelligence,cs.AI,8/1/93,8/1/93,['M. L. Ginsberg'],'M. L. Ginsberg',Because of their occasional need to return to ...,79
1,cs-9308102v1,A Market-Oriented Programming Environment and ...,Artificial Intelligence,cs.AI,8/1/93,8/1/93,['M. P. Wellman'],'M. P. Wellman',Market price systems constitute a well-underst...,119
2,cs-9309101v1,An Empirical Analysis of Search in GSAT,Artificial Intelligence,cs.AI,9/1/93,9/1/93,"['I. P. Gent', 'T. Walsh']",'I. P. Gent',We describe an extensive study of search in GS...,167
3,cs-9311101v1,The Difficulties of Learning Logic Programs wi...,Artificial Intelligence,cs.AI,11/1/93,11/1/93,"['F. Bergadano', 'D. Gunetti', 'U. Trinchero']",'F. Bergadano',As real logic programmers normally use cut (!)...,174
4,cs-9311102v1,Software Agents: Completing Patterns and Const...,Artificial Intelligence,cs.AI,11/1/93,11/1/93,"['J. C. Schlimmer', 'L. A. Hermens']",'J. C. Schlimmer',To support the goal of allowing users to recor...,187
...,...,...,...,...,...,...,...,...,...,...
136233,abs-2408.08541v1,Where is the signal in tokenization space?,Computation and Language (Natural Language Pro...,cs.CL,8/16/24,8/16/24,"['Renato Lui Geh', 'Honghua Zhang', 'Kareem Ah...",'Renato Lui Geh',Large Language Models (LLMs) are typically shi...,170
136234,abs-2408.08564v1,Collaborative Cross-modal Fusion with Large La...,Information Retrieval,cs.IR,8/16/24,8/16/24,"['Zhongzhou Liu', 'Hao Zhang', 'Kuicai Dong', ...",'Zhongzhou Liu',Despite the success of conventional collaborat...,157
136235,abs-2408.08624v1,RealMedQA: A pilot biomedical question answeri...,Computation and Language (Natural Language Pro...,cs.CL,8/16/24,8/16/24,"['Gregory Kell', 'Angus Roberts', 'Serge Umans...",'Gregory Kell',Clinical question answering systems have the p...,153
136236,abs-2408.08648v1,Understanding Enthymemes in Argument Maps: Bri...,Artificial Intelligence,cs.AI,8/16/24,8/16/24,"['Jonathan Ben-Naim', 'Victor David', 'Anthony...",'Jonathan Ben-Naim',Argument mining is natural language processing...,194


In [5]:
# we build a 'book' concatenating the titles followed by \n followed by the summary of the papers followed by \n and next title and so on 
# first we choose 100 random papers from the dataset    
df = df.sample(100)  
article_text = ''
for index, row in df.iterrows():
    article_text += row['title'] + ' \n' + row['summary'] + ' \n'    

In [6]:

# pasar todo el texto a minúscula
article_text = article_text.lower()

In [7]:

article_text[:1000]

"learning pose image manifolds using geometry-preserving gans and\n  elasticae \nthis paper investigates the challenge of learning image manifolds,\nspecifically pose manifolds, of 3d objects using limited training data. it\nproposes a dnn approach to manifold learning and for predicting images of\nobjects for novel, continuous 3d rotations. the approach uses two distinct\nconcepts: (1) geometric style-gan (geom-sgan), which maps images to\nlow-dimensional latent representations and maintains the (first-order) manifold\ngeometry. that is, it seeks to preserve the pairwise distances between base\npoints and their tangent spaces, and (2) uses euler's elastica to smoothly\ninterpolate between directed points (points + tangent directions) in the\nlow-dimensional latent space. when mapped back to the larger image space, the\nresulting interpolations resemble videos of rotating objects. extensive\nexperiments establish the superiority of this framework in learning paths on\nrotation manifold

### Elegir el tamaño del contexto

En este caso, como el modelo de lenguaje es por caracteres, todo un gran corpus
de texto puede ser considerado un documento en sí mismo y el tamaño de contexto
puede ser elegido con más libertad en comparación a un modelo de lenguaje tokenizado por palabras y dividido en documentos más acotados.

In [8]:
# seleccionamos el tamaño de contexto
max_context_size = 100

In [9]:
# Usaremos las utilidades de procesamiento de textos y secuencias de Keras
from tensorflow.keras.utils import pad_sequences # se utilizará para padding

In [10]:
# en este caso el vocabulario es el conjunto único de caracteres que existe en todo el texto
chars_vocab = set(article_text)

In [11]:
# la longitud de vocabulario de caracteres es:
len(chars_vocab)

64

In [12]:
# Construimos los dicionarios que asignan índices a caracteres y viceversa.
# El diccionario `char2idx` servirá como tokenizador.
char2idx = {k: v for v,k in enumerate(chars_vocab)}
idx2char = {v: k for k,v in char2idx.items()}

###  Tokenizar

In [13]:
# tokenizamos el texto completo
tokenized_text = [char2idx[ch] for ch in article_text]

In [14]:
tokenized_text[:1000]

[6,
 50,
 26,
 5,
 38,
 27,
 38,
 47,
 56,
 3,
 55,
 61,
 50,
 56,
 27,
 62,
 26,
 47,
 50,
 56,
 62,
 26,
 38,
 27,
 58,
 55,
 6,
 7,
 61,
 56,
 9,
 61,
 27,
 38,
 47,
 56,
 47,
 50,
 55,
 62,
 50,
 39,
 5,
 16,
 41,
 3,
 5,
 50,
 61,
 50,
 5,
 45,
 27,
 38,
 47,
 56,
 47,
 26,
 38,
 61,
 56,
 26,
 38,
 7,
 0,
 56,
 56,
 50,
 6,
 26,
 61,
 39,
 27,
 30,
 26,
 50,
 56,
 0,
 39,
 57,
 27,
 61,
 56,
 3,
 26,
 3,
 50,
 5,
 56,
 27,
 38,
 45,
 50,
 61,
 39,
 27,
 47,
 26,
 39,
 50,
 61,
 56,
 39,
 57,
 50,
 56,
 30,
 57,
 26,
 6,
 6,
 50,
 38,
 47,
 50,
 56,
 55,
 58,
 56,
 6,
 50,
 26,
 5,
 38,
 27,
 38,
 47,
 56,
 27,
 62,
 26,
 47,
 50,
 56,
 62,
 26,
 38,
 27,
 58,
 55,
 6,
 7,
 61,
 31,
 0,
 61,
 3,
 50,
 30,
 27,
 58,
 27,
 30,
 26,
 6,
 6,
 16,
 56,
 3,
 55,
 61,
 50,
 56,
 62,
 26,
 38,
 27,
 58,
 55,
 6,
 7,
 61,
 31,
 56,
 55,
 58,
 56,
 40,
 7,
 56,
 55,
 60,
 28,
 50,
 30,
 39,
 61,
 56,
 9,
 61,
 27,
 38,
 47,
 56,
 6,
 27,
 62,
 27,
 39,
 50,
 7,
 56,
 39,
 5,
 26,
 27,
 38,


### Organizando y estructurando el dataset

In [15]:
# separaremos el dataset entre entrenamiento y validación.
# `p_val` será la proporción del corpus que se reservará para validación
# `num_val` es la cantidad de secuencias de tamaño `max_context_size` que se usará en validación
p_val = 0.1
num_val = int(np.ceil(len(tokenized_text)*p_val/max_context_size))

In [16]:
# separamos la porción de texto utilizada en entrenamiento de la de validación.
train_text = tokenized_text[:-num_val*max_context_size]
val_text = tokenized_text[-num_val*max_context_size:]

In [17]:
tokenized_sentences_val = [val_text[init*max_context_size:init*(max_context_size+1)] for init in range(num_val)]

In [18]:
tokenized_sentences_train = [train_text[init:init+max_context_size] for init in range(len(train_text)-max_context_size+1)]

In [19]:
X = np.array(tokenized_sentences_train[:-1])
y = np.array(tokenized_sentences_train[1:])

In [20]:
tokenized_sentences_train = [train_text[init:init+max_context_size] for init in range(len(train_text)-max_context_size+1)]

import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_length = 100
train_text_indices = tokenized_text

input_sequences_train = []
target_sequences_train = []

for i in range(len(train_text_indices) - max_length):
    input_seq = train_text_indices[i : i + max_length]
    target_seq = train_text_indices[i + 1 : i + max_length + 1] # The next character at each step

    input_sequences_train.append(input_seq)
    target_sequences_train.append(target_seq)

X_train = np.array(input_sequences_train)
y_train = np.array(target_sequences_train)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of y_train: {y_train.shape}")

Shape of X_train: (122312, 100)
Shape of y_train: (122312, 100)


Nótese que estamos estructurando el problema de aprendizaje como *many-to-many*:

Entrada: secuencia de tokens [$x_0$, $x_1$, ..., $x_N$]

Target: secuencia de tokens [$x_1$, $x_2$, ..., $x_{N+1}$]

De manera que la red tiene que aprender que su salida deben ser los tokens desplazados en una posición y un nuevo token predicho (el N+1).

La ventaja de estructurar el aprendizaje de esta manera es que para cada token de target se propaga una señal de gradiente por el grafo de cómputo recurrente, que es mejor que estructurar el problema como *many-to-one* en donde sólo una señal de gradiente se propaga.

En este punto tenemos en la variable `tokenized_sentences` los versos tokenizados. Vamos a quedarnos con un conjunto de validación que utilizaremos para medir la calidad de la generación de secuencias con la métrica de Perplejidad.

In [21]:
X.shape

(110012, 100)

In [22]:
X[0,:10]

array([ 6, 50, 26,  5, 38, 27, 38, 47, 56,  3])

In [23]:
y.shape

(110012, 100)

In [24]:
y[0,:10]

array([50, 26,  5, 38, 27, 38, 47, 56,  3, 55])

In [25]:
vocab_size = len(chars_vocab)

# Definiendo los modelos

Pruebo los modelos sugeridos, SimpleRNN, LSTM y GRU

In [26]:
from keras.layers import Input, TimeDistributed, CategoryEncoding, SimpleRNN, Dense
from keras.models import Model, Sequential

El modelo que se propone como ejemplo consume los índices de los tokens y los transforma en vectores OHE (en este caso no entrenamos una capa de embedding para caracteres). Esa transformación se logra combinando las capas `CategoryEncoding` que transforma a índices a vectores OHE y `TimeDistributed` que aplica la capa a lo largo de la dimensión "temporal" de la secuencia.

In [27]:
model_SimpleRNN = Sequential()

model_SimpleRNN.add(TimeDistributed(CategoryEncoding(num_tokens=vocab_size, output_mode = "one_hot"),input_shape=(None,1)))
model_SimpleRNN.add(SimpleRNN(200, return_sequences=True, dropout=0.1, recurrent_dropout=0.1 ))
model_SimpleRNN.add(Dense(vocab_size, activation='softmax'))
model_SimpleRNN.compile(loss='sparse_categorical_crossentropy', optimizer='rmsprop')

model_SimpleRNN.summary()

/home/juan/anaconda3/envs/CEIA/lib/python3.12/site-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
I0000 00:00:1745447643.529848 2045576 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6103 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, None, 64)       │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, None, 200)      │        53,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, None, 64)       │        12,864 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 65,864 (257.28 KB)

 Trainable params: 65,864 (257.28 KB)

 Non-trainable params: 0 (0.00 B)

En el modelo LSTM en vez de aplicar OHE se agrega una capa de embedding a la entrada.

In [28]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dropout, Dense
from tensorflow.keras.losses import SparseCategoricalCrossentropy

model_lstm = Sequential()

# Embedding:
# input_seq_len = max_context_size --> longitud máxima de la secuencia de entrada
# input_dim = vocab_size + 1 --> tamaño del vocabulario (palabras distintas + token para fuera de vocabulario)
# output_dim = 50 --> dimensión del espacio de embedding
model_lstm.add(Embedding(input_dim=vocab_size + 1,
                    output_dim=50,
                    input_shape=(max_context_size,)))

model_lstm.add(LSTM(64, return_sequences=True))
model_lstm.add(Dropout(0.2))
model_lstm.add(LSTM(64,return_sequences=True)) # La última capa LSTM no lleva return_sequences
model_lstm.add(Dense(32, activation='relu'))

# Predicción de clasificación con softmax
# La salida vuelve al espacio de vocab_size + 1 palabras posibles
model_lstm.add(Dense(vocab_size + 1, activation='softmax'))

# Clasificación multiple categórica --> loss = SparseCategoricalCrossentropy
model_lstm.compile(loss=SparseCategoricalCrossentropy(), optimizer='rmsprop', metrics=['accuracy'])

model_lstm.summary()


/home/juan/anaconda3/envs/CEIA/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 50)        │         3,250 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 100, 64)        │        29,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100, 64)        │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100, 32)        │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 100, 65)        │         2,145 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 69,939 (273.20 KB)

 Trainable params: 69,939 (273.20 KB)

 Non-trainable params: 0 (0.00 B)

Por ultimo probamos el modelo GRU

In [29]:
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dropout, Dense
from tensorflow.keras.losses import SparseCategoricalCrossentropy


def crear_modelo_gru(vocab_size, embedding_dim, rnn_units, dropout_rate, sequence_length):
    model = Sequential([
        Embedding(input_dim=vocab_size + 1, # 
                  output_dim=embedding_dim,
                  input_shape=(sequence_length,)),
        GRU(rnn_units, return_sequences=True, dropout=dropout_rate, recurrent_dropout=dropout_rate),
        GRU(rnn_units, return_sequences=True, dropout=dropout_rate, recurrent_dropout=dropout_rate),
        Dense(vocab_size + 1, activation='softmax')
    ])
    return model

embedding_dim = 64
rnn_units = 64
dropout_rate = 0.2
sequence_length = max_context_size

model_gru = crear_modelo_gru(vocab_size, embedding_dim, rnn_units, dropout_rate, sequence_length)

model_gru.compile(loss=SparseCategoricalCrossentropy(), optimizer='rmsprop', metrics=['accuracy'])

model_gru.summary()

/home/juan/anaconda3/envs/CEIA/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 64)        │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 100, 128)       │        74,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 100, 128)       │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 100, 65)        │         8,385 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 186,113 (727.00 KB)

 Trainable params: 186,113 (727.00 KB)

 Non-trainable params: 0 (0.00 B)


### Definir el modelo

Dado que por el momento no hay implementaciones adecuadas de la perplejidad que puedan operar en tiempo de entrenamiento, armaremos un Callback *ad-hoc* que la calcule en cada epoch.

**Nota**: un Callback es una rutina gatillada por algún evento, son muy útiles para relevar datos en diferentes momentos del desarrollo del modelo. En este caso queremos hacer un cálculo cada vez que termina una epoch de entrenamiento.

In [30]:
class PplCallback(keras.callbacks.Callback):

    '''
    Este callback es una solución ad-hoc para calcular al final de cada epoch de
    entrenamiento la métrica de Perplejidad sobre un conjunto de datos de validación.
    La perplejidad es una métrica cuantitativa para evaluar la calidad de la generación de secuencias.
    Además implementa la finalización del entrenamiento (Early Stopping)
    si la perplejidad no mejora después de `patience` epochs.
    '''

    def __init__(self, val_data, history_ppl,model_name='my_model',patience=3):
      # El callback lo inicializamos con secuencias de validación sobre las cuales
      # mediremos la perplejidad
      self.val_data = val_data
      self.model_name = model_name
      self.target = []
      self.padded = []
      self.history_ppl = history_ppl
      count = 0
      self.info = []
      self.min_score = np.inf
      self.patience_counter = 0
      self.patience = patience

      # nos movemos en todas las secuencias de los datos de validación
      for seq in self.val_data:

        len_seq = len(seq)
        # armamos todas las subsecuencias
        subseq = [seq[:i] for i in range(1,len_seq)]
        self.target.extend([seq[i] for i in range(1,len_seq)])

        if len(subseq)!=0:

          self.padded.append(pad_sequences(subseq, maxlen=max_context_size, padding='pre'))

          self.info.append((count,count+len_seq))
          count += len_seq

      self.padded = np.vstack(self.padded)
      print("padded shape:", self.padded.shape)
      print("total sequences:", len(self.padded))
      print("max sequence length:", self.padded.shape[1])


    def on_epoch_end(self, epoch, logs=None):

        # en `scores` iremos guardando la perplejidad de cada secuencia
        scores = []

        predictions = self.model.predict(self.padded,verbose=0)

        # para cada secuencia de validación
        for start,end in self.info:

          # en `probs` iremos guardando las probabilidades de los términos target
          probs = [predictions[idx_seq,-1,idx_vocab] for idx_seq, idx_vocab in zip(range(start,end),self.target[start:end])]

          # calculamos la perplejidad por medio de logaritmos
          scores.append(np.exp(-np.sum(np.log(probs))/(end-start)))

        # promediamos todos los scores e imprimimos el valor promedio
        current_score = np.mean(scores)
        self.history_ppl.append(current_score)
        print(f'\n mean perplexity: {current_score} \n')

        # chequeamos si tenemos que detener el entrenamiento
        if current_score < self.min_score:
          self.min_score = current_score
          self.model.save(self.model_name+'.keras')
          print("Saved new model!")
          self.patience_counter = 0
        else:
          self.patience_counter += 1
          if self.patience_counter == self.patience:
            print("Stopping training...")
            self.model.stop_training = True


### Entrenamiento

In [31]:
y_train.shape

(122312, 100)

In [32]:
history_ppl_lstm = []
hist_lstm = model_lstm.fit(X_train, y_train, epochs=20,callbacks=[PplCallback(tokenized_sentences_val,history_ppl_lstm,model_name='model_lstm')],  batch_size=256)

padded shape: (7359, 100)
total sequences: 7359
max sequence length: 100
Epoch 1/20


I0000 00:00:1745447645.636620 2045738 cuda_dnn.cc:529] Loaded cuDNN version 90300


475/478 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1227 - loss: 3.1981
 mean perplexity: 17.960944950436154 

Saved new model!
478/478 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.1227 - loss: 3.1971
Epoch 2/20
475/478 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1728 - loss: 2.8388
 mean perplexity: 13.71062309769852 

Saved new model!
478/478 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.1730 - loss: 2.8382
Epoch 3/20
475/478 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2655 - loss: 2.5518
 mean perplexity: 11.734062364097111 

Saved new model!
478/478 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.2656 - loss: 2.5514
Epoch 4/20
475/478 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3122 - loss: 2.3802
 mean perplexity: 10.399947830783734 

Saved new model!
478/478 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.3123 - loss: 2.3800
Epoch 5/20
475/478 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.3455 - loss: 2.2588
 mean perplexity: 9.569654874905913 

Saved new model!

In [33]:
history_gru = []
hist_gru = model_gru.fit(X_train, y_train, epochs=20, callbacks=[PplCallback(tokenized_sentences_val,history_gru,model_name='model_GRU')], batch_size=256)

padded shape: (7359, 100)
total sequences: 7359
max sequence length: 100
Epoch 1/20
478/478 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step - accuracy: 0.1971 - loss: 2.8932
 mean perplexity: 8.913219873576988 

Saved new model!
478/478 ━━━━━━━━━━━━━━━━━━━━ 83s 169ms/step - accuracy: 0.1972 - loss: 2.8924
Epoch 2/20
478/478 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - accuracy: 0.4225 - loss: 1.9899
 mean perplexity: 6.390208644843201 

Saved new model!
478/478 ━━━━━━━━━━━━━━━━━━━━ 80s 167ms/step - accuracy: 0.4226 - loss: 1.9897
Epoch 3/20
478/478 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step - accuracy: 0.5042 - loss: 1.6885
 mean perplexity: 5.706676091618814 

Saved new model!
478/478 ━━━━━━━━━━━━━━━━━━━━ 79s 166ms/step - accuracy: 0.5042 - loss: 1.6884
Epoch 4/20
478/478 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step - accuracy: 0.5432 - loss: 1.5419
 mean perplexity: 5.315527888712283 

Saved new model!
478/478 ━━━━━━━━━━━━━━━━━━━━ 78s 164ms/step - accuracy: 0.5432 - loss: 1.5419
Epoch 5/20
478/478 ━━━━━━━━━━━━━━━━━━━━ 0s 

In [34]:
# fiteamos, nótese el agregado del callback con su inicialización. El batch_size lo podemos seleccionar a mano
# en general, lo mejor es escoger el batch más grande posible que minimice el tiempo de cada época.
# En la variable `history_ppl` se guardarán los valores de perplejidad para cada época.
history_ppl = []
hist = model_SimpleRNN.fit(X_train, y_train, epochs=20, callbacks=[PplCallback(tokenized_sentences_val,history_ppl,model_name='model_SimpleRNN')], batch_size=256)

padded shape: (7359, 100)
total sequences: 7359
max sequence length: 100
Epoch 1/20


I0000 00:00:1745449633.793899 2045738 service.cc:152] XLA service 0x7553f4f41390 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1745449633.794610 2045738 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Laptop GPU, Compute Capability 8.9
2025-04-23 20:07:14.086450: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-04-23 20:07:14.137774: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
2025-04-23 20:07:15.476082: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_33', 156 bytes spill stores, 156 bytes spill loads

2025-04-23 20:07:15.555564: I external/local_xl

 19/478 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 3.5195

I0000 00:00:1745449640.605986 2045738 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


475/478 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 2.7741

2025-04-23 20:07:24.288805: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
2025-04-23 20:07:25.898275: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_3482', 4 bytes spill stores, 4 bytes spill loads

2025-04-23 20:07:25.912454: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_33', 160 bytes spill stores, 160 bytes spill loads

2025-04-23 20:07:25.971749: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_33', 608 bytes spill stores, 588 bytes spill loads

2025-04-23 20:07:26.083566: I external/local_xla/xla/stream_execu

478/478 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 2.7724
 mean perplexity: 8.983868597230803 

Saved new model!
478/478 ━━━━━━━━━━━━━━━━━━━━ 26s 35ms/step - loss: 2.7719
Epoch 2/20
475/478 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 2.1845
 mean perplexity: 6.999082621777662 

Saved new model!
478/478 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 2.1840
Epoch 3/20
474/478 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1.9830
 mean perplexity: 6.04412730400147 

Saved new model!
478/478 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 1.9826
Epoch 4/20
473/478 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1.8525
 mean perplexity: 5.504263418486715 

Saved new model!
478/478 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 1.8522
Epoch 5/20
473/478 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1.7738
 mean perplexity: 5.1687014837286 

Saved new model!
478/478 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 1.7736
Epoch 6/20
475/478 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1.7236
 mean perplexity: 4.9379996822462005 

Saved new model

In [1]:
import matplotlib.pyplot as plt
import seaborn as sns

# Entrenamiento
epoch_count = range(1, len(history_ppl_lstm) + 1)
sns.lineplot(x=epoch_count,  y=history_ppl_lstm,label='lstm',legend=True)
epoch_count = range(1, len(history_ppl) + 1)
sns.lineplot(x=epoch_count,  y=history_ppl,label='simpleRNN',legend=True)
epoch_count = range(1, len(history_gru) + 1)

sns.lineplot(x=epoch_count,  y=history_gru,label='GRU',legend=True)

plt.show()
plt.savefig('perplexity.png')

NameError: name 'history_ppl_lstm' is not defined

In [39]:
# Cargamos el mejor modelo guardado del entrenamiento para hacer inferencia
model_simpleRNN = keras.models.load_model('model_SimpleRNN.keras')
model_gru = keras.models.load_model('model_GRU.keras')
model_lstm = keras.models.load_model('model_lstm.keras')


### Predicción del próximo caracter

In [118]:
# Se puede usar gradio para probar el modelo
# Gradio es una herramienta muy útil para crear interfaces para ensayar modelos
# https://gradio.app/

!pip install -q gradio

In [40]:
import gradio as gr

def model_response(human_text):

    # Encodeamos
    encoded = [char2idx[ch] for ch in human_text.lower() ]
    # Si tienen distinto largo
    encoded = pad_sequences([encoded], maxlen=max_context_size, padding='pre')

    # Predicción softmax
    y_hat = np.argmax(model.predict(encoded)[0,-1,:])


    # Debemos buscar en el vocabulario el caracter
    # que corresopnde al indice (y_hat) predicho por le modelo
    out_word = ''
    out_word = idx2char[y_hat]

    # Agrego la palabra a la frase predicha
    return human_text + out_word

iface = gr.Interface(
    fn=model_response,
    inputs=["textbox"],
    outputs="text")

# iface.launch(debug=True)

### Generación de secuencias

In [41]:
def generate_seq(model, seed_text, max_length, n_words):
    """
        Exec model sequence prediction

        Args:
            model (keras): modelo entrenado
            seed_text (string): texto de entrada (input_seq)
            max_length (int): máxima longitud de la sequencia de entrada
            n_words (int): números de caracteres a agregar a la sequencia de entrada
        returns:
            output_text (string): sentencia con las "n_words" agregadas
    """
    output_text = seed_text
	# generate a fixed number of words
    for _ in range(n_words):
		# Encodeamos
        encoded = [char2idx[ch] for ch in output_text.lower() ]
		# Si tienen distinto largo
        encoded = pad_sequences([encoded], maxlen=max_length, padding='pre')

		# Predicción softmax
        y_hat = np.argmax(model.predict(encoded,verbose=0)[0,-1,:])
		# Vamos concatenando las predicciones
        out_word = ''

        out_word = idx2char[y_hat]

		# Agrego las palabras a la frase predicha
        output_text += out_word
    return output_text

In [42]:
input_text='modern artificial'

generate_seq(model_lstm, input_text, max_length=max_context_size, n_words=100)

'modern artificial and the desent and the desent and the desent and the desent and the desent and the desent and the d'

In [43]:
generate_seq(model_gru, input_text, max_length=max_context_size, n_words=100)


'modern artificial and the proposed and the proposed and the proposed and the proposed and the proposed and the propos'

In [44]:
generate_seq(model_simpleRNN, input_text, max_length=max_context_size, n_words=100)


'modern artificial experiments on the proposed as a sense vector of the and results on the proposed as a sense vector '

###  Beam search y muestreo aleatorio

In [45]:
# funcionalidades para hacer encoding y decoding

def encode(text,max_length=max_context_size):

    encoded = [char2idx[ch] for ch in text]
    encoded = pad_sequences([encoded], maxlen=max_length, padding='pre')

    return encoded

def decode(seq):
    return ''.join([idx2char[ch] for ch in seq])

In [46]:
from scipy.special import softmax

# función que selecciona candidatos para el beam search
def select_candidates(pred,num_beams,vocab_size,history_probs,history_tokens,temp,mode):

  # colectar todas las probabilidades para la siguiente búsqueda
  pred_large = []

  for idx,pp in enumerate(pred):
    pred_large.extend(np.log(pp+1E-10)+history_probs[idx])

  pred_large = np.array(pred_large)

  # criterio de selección
  if mode == 'det':
    idx_select = np.argsort(pred_large)[::-1][:num_beams] # beam search determinista
  elif mode == 'sto':
    idx_select = np.random.choice(np.arange(pred_large.shape[0]), num_beams, p=softmax(pred_large/temp)) # beam search con muestreo aleatorio
  else:
    raise ValueError(f'Wrong selection mode. {mode} was given. det and sto are supported.')

  # traducir a índices de token en el vocabulario
  new_history_tokens = np.concatenate((np.array(history_tokens)[idx_select//vocab_size],
                        np.array([idx_select%vocab_size]).T),
                      axis=1)

  # devolver el producto de las probabilidades (log) y la secuencia de tokens seleccionados
  return pred_large[idx_select.astype(int)], new_history_tokens.astype(int)


def beam_search(model,num_beams,num_words,input,temp=1,mode='det'):

    # first iteration

    # encode
    encoded = encode(input)

    # first prediction
    y_hat = model.predict(encoded,verbose=0)[0,-1,:]

    # get vocabulary size
    vocab_size = y_hat.shape[0]

    # initialize history
    history_probs = [0]*num_beams
    history_tokens = [encoded[0]]*num_beams

    # select num_beams candidates
    history_probs, history_tokens = select_candidates([y_hat],
                                        num_beams,
                                        vocab_size,
                                        history_probs,
                                        history_tokens,
                                        temp,
                                        mode)

    # beam search loop
    for i in range(num_words-1):

      preds = []

      for hist in history_tokens:

        # actualizar secuencia de tokens
        input_update = np.array([hist[i+1:]]).copy()

        # predicción
        y_hat = model.predict(input_update,verbose=0)[0,-1,:]

        preds.append(y_hat)

      history_probs, history_tokens = select_candidates(preds,
                                                        num_beams,
                                                        vocab_size,
                                                        history_probs,
                                                        history_tokens,
                                                        temp,
                                                        mode)

    return history_tokens[:,-(len(input)+num_words):]

### Predicciones con beamsearch deterministico

In [47]:
# predicción con beam search
salidas = beam_search(model_lstm,num_beams=5,num_words=50,input="modern",mode='det')
decode(salidas[0])

'moderning the proposed the proposed the proposed and the'

In [48]:
# predicción con beam search
salidas = beam_search(model_gru,num_beams=5,num_words=50,input="modern",mode='det')
decode(salidas[0])

'modern. we propose a compared to the performance of the '

In [49]:
# predicción con beam search
salidas = beam_search(model_simpleRNN,num_beams=5,num_words=50,input="modern",mode='det')
decode(salidas[0])

'modern models and effective learning method on the propo'

### Predicciones con beamsearch estocastico
Ahora podemos jugar un poco con la temperatura

#### Temperatura baja

In [50]:
# predicción con beam search
salidas = beam_search(model_lstm,num_beams=5,num_words=50,input="modern",temp=2,mode='sto')
decode(salidas[0])

'moderner on the\nnobulatives. contrast matle and man our '

In [51]:
# predicción con beam search
salidas = beam_search(model_gru,num_beams=5,num_words=50,input="modern",temp=2,mode='sto')
decode(salidas[0])# predicción con beam search


'modern callenges: similarity output we propose a low dis'

In [52]:
salidas = beam_search(model_simpleRNN,num_beams=5,num_words=50,input="modern",temp=2,mode='sto')
decode(salidas[0])

'modernes to all sea sing of the advms used can be addaps'

#### Temperatura alta

In [53]:
# predicción con beam search
salidas = beam_search(model_lstm,num_beams=5,num_words=50,input="modern",temp=10,mode='det')
decode(salidas[0])

'moderning the proposed the proposed the proposed and the'

In [55]:
# predicción con beam search
salidas = beam_search(model_gru,num_beams=5,num_words=50,input="modern",temp=10,mode='det')
decode(salidas[0])

'modern. we propose a compared to the performance of the '

In [54]:
salidas = beam_search(model_simpleRNN,num_beams=5,num_words=50,input="modern",temp=10,mode='det')
decode(salidas[0])

'modern models and effective learning method on the propo'

# Conclusiones

La evolucion de la estimacion de perplejidad con el callback provisto parece bastante razonable en todos los casos, se ve que los modelos aprenden.

Los modelos entrenados infieren en su mayoria palabras reales en el modo de inferencia 'deterministico' o usando softmax, pero no logran producir muchos caracteres. 

Esto en principio yo se lo atribuiria a los datos, ya que hay muchos terminos que se repiten en los abstracts de papers y por lo tanto el dataset esta bastante sesgado hacia esos terminos o expresiones, comose puede apreciar en los distintos resultados que pude obtener.

Sin embargo podria probar tambien agrandando el contexto utilizado en entrenamiento (100 caracteres) o bien las epocas de entrenamiento (50 o 100).  Quizas tambien agregando mas neuronas en las capas recurrentes se puedan obtener mejores resultados.

Por ultimo en el experimento de la inferencia utilizando beams earch estocastico, pude comprobar que la temperatura influye en la diversidad de las predicciones. Incluso generando secuencias más lógicas / interesantes que con el metodo deterministico.
